# Aplicación a la especie foco

Embudo de compuestos huérfanos con actividad fenotípica contra la especie foco
(`tdr.SP_FOCO`, un protozoo parásito) y
propuesta de blancos para los que el método puede tratar:

  1. compuestos con bioactividad fenotípica positiva contra la especie foco;
  2. de esos, los que **no** tienen ningún enlace de bioactividad a un blanco
     proteico (huérfanos en el sentido del paper);
  3. de esos, los tratables: con al menos un vecino químico con blanco conocido.

Para cada tratable se arma la semilla con su vecindario y se propaga; se conservan
las proteínas con `rG ≤ r*G`, el corte estimado en el notebook `03`.

Fuente: `reproducir_v6/aplicacion_especie/`.
Salidas: `04_embudo.csv`, `04_compuestos.csv`, `04_sugerencias.csv`.

## Imports

In [ ]:
import sys, os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # antes de numpy: un hilo por proceso

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

V4 = "/home/ggiordano/TDR/TDR_2026_v4"
sys.path.insert(0, f"{V4}/comun")
sys.path.insert(0, f"{V4}/huerfanas")
%load_ext autoreload
%autoreload 2
import tdr, nucleo as nf
import funciones_huerfanas as fh              # el .py de esta carpeta

SALIDAS = tdr.out("huerfanas")
FIGURAS = SALIDAS / "figuras"
NB      = "04"                             # numero de este notebook: prefija todo lo que genere
n_core  = 20

plt = tdr.estilo()
SALIDAS

## Datos

In [ ]:
# cluster_consistent=True: `huerfanas/` filtra los positivos/negativos
# inconsistentes a nivel cluster, como orphan_drugs_v4.ipynb.
crudo = tdr.cargar_db(anotaciones=True, quimica=True, fenotipo=True, cluster_consistent=True)

# README §7.3: el filtro de promiscuidad se APLICA antes de armar ninguna semilla.
# La lista sale de analiceDB/03; si falta, el error dice qué correr.
promiscuos = tdr.compuestos_promiscuos()
datos = tdr.filtrar_capa_quimica(crudo, promiscuos)

HUERFANAS = tdr.out("huerfanas")
rg_star = float(pd.read_csv(HUERFANAS / "03_rg_estrella.csv")["r_g_estrella"].iloc[0])
optimos = fh.parametros_optimos()
print(f"corte r*G = {rg_star:.0f}  ·  parámetros de sp{tdr.SP_FOCO}: {optimos[tdr.SP_FOCO]}")

## Acondicionamiento

In [ ]:
SP = tdr.SP_FOCO   # protozoo parásito; ver tdr.SPECIES
embudo, compuestos = fh.embudo_especie(datos, sp=SP)
tratables = compuestos.loc[compuestos["tratable"], "drug_id"].values
print(embudo.to_string(index=False))

fig = fh.fig_embudo(embudo, plt)
tdr.guardar(fig, f"{N
B}_f01_embudo", FIGURAS)

## Corrida

In [ ]:
# celda de corrida: el ciclo se lee aca
ctx = fh.preparar_especie(datos, SP, optimos[SP], pd.DataFrame({"sp_id": []}))
posdt_sp = datos.posdt.merge(datos.st[["target_id", "sp_id"]], on="target_id", how="left")
fh.fijar_contexto(datos, posdt_sp, ctx)

sugerencias = []
for cid in tratables:
    seed = fh.semilla_de_droga(datos, cid, datos.posdt[["drug_id", "target_id"]],
                               consolidar=True)
    if seed is None or seed.empty:
        continue
    rnk = tdr.propagar(datos.sta, seed, ctx["cat_rs"], beta=ctx["beta"],
                       lambda_=ctx["lambda_"], gamma=ctx["gamma"])
    rnk["rG"] = rnk["score"].rank(ascending=False, method="average")
    top = rnk[(rnk["rG"] <= rg_star) & rnk["target_id"].astype(str).isin(ctx["sp_targets"])]
    sugerencias.append(top.assign(drug_id=cid))

sug = pd.concat(sugerencias, ignore_index=True) if sugerencias else pd.DataFrame()
embudo.to_csv(SALIDAS / f"{NB}_embudo.csv", index=False)
compuestos.to_csv(SALIDAS / f"{NB}_compuestos.csv", index=False)
sug.to_csv(SALIDAS / f"{NB}_sugerencias.csv", index=False)
fh.escribir_meta(SALIDAS, NB, notebook="04_aplicacion_especie.ipynb",
                 params={"especie": SP, "r_g_estrella": rg_star, "optimos": optimos[SP]},
                 n_core=n_core, filtro_promiscuidad=True)

# Resultados

In [ ]:
embudo     = pd.read_csv(SALIDAS / f"{NB}_embudo.csv")
compuestos = pd.read_csv(SALIDAS / f"{NB}_compuestos.csv")
sug        = pd.read_csv(SALIDAS / f"{NB}_sugerencias.csv")
print(f"{sug['drug_id'].nunique()} compuestos con sugerencias, "
      f"{sug['target_id'].nunique()} blancos propuestos")
sug.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.5), tight_layout=True)
ax.hist(sug["rG"], bins=np.arange(1, rg_star + 2), color=tdr.S1)
ax.set_xlabel("rG de la sugerencia")
ax.set_ylabel("pares compuesto-blanco")
ax.set_title(f"Sugerencias por debajo de r*G = {rg_star:.0f}")
tdr.guardar(fig, f"{NB}_f02_rg_sugerencias", FIGURAS)

In [ ]:
# Familias de blancos más sugeridas: los casos de estudio del paper (Figs 5, 7, 8)
familias = tdr.anotar_targets(sug["target_id"].unique(), datos.sta)
top = (sug.merge(familias, on="target_id", how="left")
       .groupby("anotaciones")["drug_id"].nunique()
       .sort_values(ascending=False).head(15))
fig, ax = plt.subplots(figsize=(7, 4), tight_layout=True)
ax.barh(range(len(top)), top.values, color=tdr.S1)
ax.set_yticks(range(len(top)))
ax.set_yticklabels([t[:60] for t in top.index], fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("compuestos huérfanos que la reciben")
tdr.guardar(fig, f"{NB}_f03_familias", FIGURAS)